In [ ]:
%cd /content
!rm -rf worldindex
!git clone https://github.com/RivaanRanawat/worldindex.git worldindex
%cd /content/worldindex

In [ ]:
!curl -sSL https://install.python-poetry.org | python3 -
import os
os.environ["PATH"] += ":/root/.local/bin"
!poetry install --with video

In [ ]:
%%bash

export WORLDINDEX_REAL_EXTRACTION_CLIP_LIMIT=100
poetry run python scripts/pipeline.py extract --config config/gpu_pipeline_droid.yaml


In [ ]:
!poetry run python scripts/pipeline.py run --config config/gpu_pipeline_droid.yaml

In [ ]:
!poetry run python scripts/pipeline.py validate --config config/gpu_pipeline_droid.yaml 

In [ ]:
%%bash
cd /content/worldindex

PORT=8000

OLD_PID=$(lsof -ti tcp:${PORT} -sTCP:LISTEN 2>/dev/null | head -n 1 || true)
if [ -n "${OLD_PID}" ]; then
  kill "${OLD_PID}" 2>/dev/null || true
  sleep 2
fi

rm -f /content/worldindex/server.pid

nohup poetry run python scripts/pipeline.py serve --config config/gpu_pipeline_droid.yaml \
  > /content/worldindex/server.log 2>&1 &

echo $! > /content/worldindex/server.pid
sleep 5

echo "server pid: $(cat /content/worldindex/server.pid)"
tail -n 40 /content/worldindex/server.log

In [ ]:
!curl -s http://127.0.0.1:8000/health

In [ ]:
%%bash
cd /content/worldindex
mkdir -p tmp

curl -L "https://hackaday.com/wp-content/uploads/2025/08/Running-AI-robotics-experiments-at-home-with-LeRobot-and-SO-ARM100-35-25-screenshot-banner.png" \
  -o tmp/img1.jpg

curl -L "https://images.ctfassets.net/jdtwqhzvc2n1/4UkU5lrsHXuvAmTJK07nr6/b9fc657475c88c3dc25611ea5635e59d/Screen-Shot-2024-01-05-at-1.40.51-PM.png" \
  -o tmp/img2.jpg

ls -lh tmp/

In [ ]:
%%bash
cd /content/worldindex
curl -s -F "image=@tmp/img2.jpg;type=image/jpeg" \
  "http://127.0.0.1:8000/search/image?top_k=5&coarse_k=10" | python -m json.tool